In [0]:
# Cell 1: Setup Access to Storage
storage_name = "ststockanalyticskiran" # UPDATE THIS
storage_key = "YOUR_AZURE_STORAGE_KEY" # UPDATE THIS

spark.conf.set(
    f"fs.azure.account.key.{storage_name}.dfs.core.windows.net",
    storage_key
)

# Cell 2: Read Bronze Data
# We read all CSVs at once. The 'Ticker' column we added earlier keeps them identified.
bronze_path = f"abfss://bronze@{storage_name}.dfs.core.windows.net/*.csv"

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

# Cell 3: Transformation Logic
from pyspark.sql.functions import col, avg, to_date, lit
from pyspark.sql.window import Window

# Fix Date format if needed and cast types
df_clean = df_raw.withColumn("Date", to_date(col("Date"))) \
                 .withColumn("Close", col("Close").cast("double")) \
                 .withColumn("Volume", col("Volume").cast("long"))

# Define Windows for Moving Averages
# SMA 50: Previous 50 days
w_50 = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-49, 0)
# SMA 200: Previous 200 days
w_200 = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-199, 0)

df_enriched = df_clean.withColumn("SMA_50", avg("Close").over(w_50)) \
                      .withColumn("SMA_200", avg("Close").over(w_200))

# Calculate Volatility (Daily Returns)
# Simple way: (Close - Open) / Open
df_final = df_enriched.withColumn("Daily_Return", (col("Close") - col("Open")) / col("Open"))

# Cell 4: Write to Gold (Parquet Format)
# We coalesce(1) to get a single file per partition for easier reading in Power BI (optional but helpful for small data)
gold_path = f"abfss://gold@{storage_name}.dfs.core.windows.net/stock_facts"

df_final.write.mode("overwrite").parquet(gold_path)

print(f"Successfully processed data to {gold_path}")

Successfully processed data to abfss://gold@ststockanalyticskiran.dfs.core.windows.net/stock_facts


In [0]:
import requests
import json
import time
import datetime
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

print("Starting AI Portfolio Analysis...")

# 1. Setup API Details (PASTE YOUR KEY HERE)
API_KEY = "YOUR_GEMINI_API_KEY"
url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}"
headers = {'Content-Type': 'application/json'}

# 2. Get a list of all unique companies (and filter out dirty 'None' rows!)
tickers_df = df_enriched.filter(col("Ticker").isNotNull()).select("Ticker").distinct().collect()
tickers_list = [row["Ticker"] for row in tickers_df]

print(f"Found {len(tickers_list)} valid companies to analyze: {tickers_list}")

# 3. Create an empty list to store all the AI's advice
ai_results = []
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

# 4. Loop through EVERY company and ask the AI for advice
for ticker in tickers_list:
    # Get the latest data for this specific ticker
    latest_data = df_enriched.filter(df_enriched.Ticker == ticker).orderBy(col("Date").desc()).first()
    
    # The Upgraded Prompt: Asking for actionable suggestions (Buy/Hold/Sell)
    prompt_text = f"Act as a professional financial analyst. {ticker}'s latest closing price is {round(latest_data['Close'], 2)}. The 50-day SMA is {round(latest_data['SMA_50'], 2)} and the 200-day SMA is {round(latest_data['SMA_200'], 2)}. In exactly 2 short sentences, tell me if this is a bullish or bearish signal, and give a specific recommendation (e.g., Buy, Hold, or Sell) to maximize portfolio growth."
    
    payload = {"contents": [{"parts": [{"text": prompt_text}]}]}
    
    # Fault Tolerance: Try up to 3 times per stock
    for attempt in range(3):
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        response_data = response.json()
        
        if 'candidates' in response_data:
            insight = response_data['candidates'][0]['content']['parts'][0]['text']
            ai_results.append((ticker, current_date, insight))
            print(f"✅ AI analyzed {ticker} successfully.")
            break
        elif response.status_code == 503:
            print(f"⚠️ Server busy. Retrying {ticker}...")
            time.sleep(5)
        else:
            print(f"❌ Error analyzing {ticker}")
            break
            
    # Pause for 2 seconds before the next stock so Google doesn't block us for going too fast
    time.sleep(2) 

# 5. Save EVERYTHING to the Azure Gold Layer
schema = StructType([
    StructField("Ticker", StringType(), True),
    StructField("Date_Generated", StringType(), True),
    StructField("AI_Insight", StringType(), True)
])

ai_df = spark.createDataFrame(ai_results, schema)

# PASTE YOUR STORAGE ACCOUNT NAME HERE
gold_path = f"abfss://gold@{storage_name}.dfs.core.windows.net/ai_insights"
ai_df.write.mode("overwrite").parquet(gold_path)
print(f"\n🚀 ALL DONE! Portfolio Insights saved to Azure: {gold_path}")

Starting AI Portfolio Analysis...
Found 6 valid companies to analyze: ['GOOGL', 'NVDA', 'AAPL', 'MSFT', 'TSLA', 'AMZN']
✅ AI analyzed GOOGL successfully.
✅ AI analyzed NVDA successfully.
✅ AI analyzed AAPL successfully.
✅ AI analyzed MSFT successfully.
✅ AI analyzed TSLA successfully.
✅ AI analyzed AMZN successfully.

🚀 ALL DONE! Portfolio Insights saved to Azure: abfss://gold@ststockanalyticskiran.dfs.core.windows.net/ai_insights
